# Notebook 01 — Exploratory Data Analysis

**Project**: Credit Card Fraud Detection  
**Dataset**: ULB Machine Learning Group — 284,807 European card transactions (Sept 2013)  
**Goal**: Understand the data distribution, confirm the class imbalance, and identify which features discriminate frauds from genuine transactions.

All figures are saved to `reports/figures/` at 150 dpi.

In [1]:
import sys, pathlib
ROOT = pathlib.Path().resolve().parents[0] if pathlib.Path().resolve().name == 'notebooks' else pathlib.Path().resolve()
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_raw, audit, print_audit
from features import add_hour_feature, V_COLS

FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Dark-theme plotting style ────────────────────────────────────────────────
STYLE = {
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
    'grid.linewidth': 0.6, 'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'font.family': 'DejaVu Sans', 'font.size': 11,
}
plt.rcParams.update(STYLE)

def save(fig, name):
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'Saved: {path}')

print('Imports OK')

Imports OK


## 1. Data Audit

In [2]:
df = load_raw()
info = audit(df)
print_audit(info)
df.head(3)

DATA AUDIT
  Rows        : 284,807
  Columns     : 31
  Missing vals: 0
  Duplicates  : 1081
  Genuine     : 284,315
  Fraud       : 492
  Fraud rate  : 0.1727 %


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0


### Findings
- The dataset has **284,807 rows** and **31 columns** (Time, V1-V28, Amount, Class).
- **No missing values** — confirmed by audit.
- **Duplicate rows** may exist; we drop them in the modelling pipeline.
- Class imbalance: only **~0.17%** of transactions are fraudulent.

In [3]:
print('dtypes:\n', df.dtypes)
print('\nDescriptive statistics (Amount & Time):')
df[['Time', 'Amount']].describe()

dtypes:
 Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Descriptive statistics (Amount & Time):


,Time,Amount
count,284807.000000,284807.000000
mean,94813.859575,88.349619
std,47488.145955,250.120109
min,0.000000,0.000000
25%,54201.500000,5.600000
50%,84692.000000,22.000000
75%,139320.500000,77.165000
max,172792.000000,25691.160000


## 2. Class Imbalance

In [4]:
counts = df['Class'].value_counts()
labels = ['Genuine (0)', 'Fraud (1)']
colors = ['#58a6ff', '#f85149']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(labels, counts.values, color=colors, edgecolor='#21262d', linewidth=0.8)
axes[0].set_yscale('log')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Class Distribution (log scale)')
for i, v in enumerate(counts.values):
    axes[0].text(i, v * 1.5, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(
    counts.values, labels=labels, colors=colors,
    autopct='%1.2f%%', startangle=90,
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2},
    textprops={'color': '#c9d1d9'},
)
axes[1].set_title('Class Proportion')

fig.suptitle('Credit Card Fraud — Extreme Class Imbalance', fontsize=14, y=1.01)
save(fig, 'class_imbalance.png')
plt.show()

Saved: D:\Projects\Credit-Card-Fraud-Detection\reports\figures\class_imbalance.png


**Key insight**: Only 492 frauds out of 284,807 transactions (0.172%). A classifier predicting *everything* as genuine achieves ~99.83% accuracy — which is why **accuracy is useless here** and we use PR-AUC instead.

## 3. Amount Distribution by Class

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for cls, color, label in [(0, '#58a6ff', 'Genuine'), (1, '#f85149', 'Fraud')]:
    subset = df[df['Class'] == cls]['Amount']
    axes[0].hist(subset + 1, bins=60, alpha=0.7, color=color,
                 label=label, edgecolor='none')
    axes[1].hist(subset + 1, bins=60, alpha=0.7, color=color,
                 label=label, edgecolor='none')

axes[0].set_xlabel('Amount + 1'); axes[0].set_ylabel('Count')
axes[0].set_title('Amount Distribution (linear)')
axes[0].legend()

axes[1].set_xscale('log'); axes[1].set_xlabel('Amount + 1 (log scale)')
axes[1].set_ylabel('Count'); axes[1].set_yscale('log')
axes[1].set_title('Amount Distribution (log-log scale)')
axes[1].legend()

fig.suptitle('Transaction Amount by Class', fontsize=13)
save(fig, 'amount_distribution.png')
plt.show()

print(df.groupby('Class')['Amount'].describe().T)

Saved: D:\Projects\Credit-Card-Fraud-Detection\reports\figures\amount_distribution.png
Class              0            1
count  284315.000000   492.000000
mean       88.291022   122.211321
std       250.105092   256.683288
min         0.000000     0.000000
25%         5.650000     1.000000
50%        22.000000     9.250000
75%        77.050000   105.890000
max     25691.160000  2125.870000


**Finding**: Fraudulent transactions tend to be smaller amounts on average (median ~22 vs ~76 for genuine) but reach up to ~2,126. Many frauds cluster below 100 units, consistent with card-testing behaviour.

## 4. Fraud Rate by Hour of Day

In [6]:
df_h = add_hour_feature(df)

hourly = df_h.groupby('Hour')['Class'].agg(['sum', 'count'])
hourly.columns = ['fraud_count', 'total']
hourly['fraud_rate'] = hourly['fraud_count'] / hourly['total']

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].bar(hourly.index, hourly['total'], color='#58a6ff', alpha=0.8, label='All transactions')
axes[0].bar(hourly.index, hourly['fraud_count'], color='#f85149', alpha=0.9, label='Frauds')
axes[0].set_ylabel('Count'); axes[0].legend()
axes[0].set_title('Transaction Volume and Fraud Count by Hour of Day')

axes[1].plot(hourly.index, hourly['fraud_rate'] * 100, 'o-',
             color='#f0883e', lw=2, ms=6)
axes[1].set_xlabel('Hour of Day'); axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate (%) by Hour')
axes[1].set_xticks(range(24))
axes[1].grid(True, alpha=0.4)

fig.tight_layout()
save(fig, 'fraud_rate_by_hour.png')
plt.show()
print(hourly)

Saved: D:\Projects\Credit-Card-Fraud-Detection\reports\figures\fraud_rate_by_hour.png
      fraud_count  total  fraud_rate
Hour                                
0.0             6   7695    0.000780
1.0            10   4220    0.002370
2.0            57   3328    0.017127
3.0            17   3492    0.004868
4.0            23   2209    0.010412
5.0            11   2990    0.003679
6.0             9   4101    0.002195
7.0            23   7243    0.003175
8.0             9  10276    0.000876
9.0            16  15838    0.001010
10.0            8  16598    0.000482
11.0           53  16856    0.003144
12.0           17  15420    0.001102
13.0           17  15365    0.001106
14.0           23  16570    0.001388
15.0           26  16461    0.001579
16.0           22  16453    0.001337
17.0           29  16166    0.001794
18.0           33  17039    0.001937
19.0           19  15649    0.001214
20.0           18  16756    0.001074
21.0           16  17703    0.000904
22.0            9  15441  

**Finding**: Fraud rates appear higher in the early morning hours (roughly 0–6 h) when transaction volume drops but fraudsters may be more active. However, with only 492 frauds the per-hour counts are very small, so the pattern has high variance.

## 5. Most Class-Separating V Features

In [7]:
# Rank V-features by absolute difference in median between classes
medians = df.groupby('Class')[V_COLS].median()
sep = (medians.loc[1] - medians.loc[0]).abs().sort_values(ascending=False)
top_features = sep.head(10).index.tolist()
print('Top 10 separating V-features (|median_fraud - median_genuine|):')
print(sep.head(10))

Top 10 separating V-features (|median_fraud - median_genuine|):
V14    6.781667
V12    5.644208
V3     5.257415
V17    5.238115
V10    4.486953
V4     4.199552
V11    3.621142
V16    3.617172
V7     3.075540
V2     2.653799
dtype: float64


In [8]:
# KDE plots for top 6 features
top6 = top_features[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(top6):
    for cls, color, label in [(0, '#58a6ff', 'Genuine'), (1, '#f85149', 'Fraud')]:
        data = df[df['Class'] == cls][feat].dropna()
        axes[i].hist(data, bins=60, alpha=0.5, color=color, label=label,
                     density=True, edgecolor='none')
    axes[i].set_title(f'{feat} distribution')
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Density')

fig.suptitle('Top 6 Most Discriminating Features (Fraud vs Genuine)', fontsize=13, y=1.01)
fig.tight_layout()
save(fig, 'top_v_feature_distributions.png')
plt.show()

Saved: D:\Projects\Credit-Card-Fraud-Detection\reports\figures\top_v_feature_distributions.png


**Finding**: Features like V14, V4, V11, V12 show clear distributional shifts between genuine and fraudulent transactions. Since these are PCA components, we cannot assign specific business meaning — but their separating power makes them valuable model inputs.

## 6. Correlation of Features with Class

In [9]:
corr_with_class = df.drop(columns=['Time']).corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(8, 9))
colors_bar = ['#f85149' if c > 0 else '#58a6ff' for c in corr_with_class]
ax.barh(corr_with_class.index, corr_with_class.values, color=colors_bar, edgecolor='none')
ax.axvline(0, color='#8b949e', lw=0.8)
ax.set_xlabel('Pearson Correlation with Class')
ax.set_title('Feature Correlation with Fraud Label')
ax.grid(axis='x', alpha=0.3)

save(fig, 'feature_correlation.png')
plt.show()

print('Most positively correlated with fraud:')
print(corr_with_class.tail(5))
print('\nMost negatively correlated with fraud:')
print(corr_with_class.head(5))

Saved: D:\Projects\Credit-Card-Fraud-Detection\reports\figures\feature_correlation.png
Most positively correlated with fraud:
V19    0.034783
V21    0.040413
V2     0.091289
V4     0.133447
V11    0.154876
Name: Class, dtype: float64

Most negatively correlated with fraud:
V17   -0.326481
V14   -0.302544
V12   -0.260593
V10   -0.216883
V16   -0.196539
Name: Class, dtype: float64


## 7. Summary

| Finding | Implication |
|---------|-------------|
| 0.172% fraud rate | Accuracy is a useless metric; use PR-AUC |
| No missing values | No imputation needed |
| Amount needs scaling | StandardScaler on Amount and Time |
| V14, V4, V11 are most separating | Model will likely rely on these |
| Hour pattern (small n) | Interesting but noisy; Time is in the feature set |

**Next**: `02_modeling.ipynb` for the full modelling pipeline.